# SQLAlchemy: sync and async, side by side

The same statements, the same models, two ways of talking to the database. Each section shows
one idea in the sync API and then in the async API, so the differences stay visible.

The short version:

| | sync | async |
|---|---|---|
| engine | `create_engine(url)` | `create_async_engine(url)` |
| connection / transaction | `with engine.connect()` / `engine.begin()` | `async with engine.connect()` / `engine.begin()` |
| session | `Session(engine)`, `sessionmaker` | `AsyncSession(engine)`, `async_sessionmaker` |
| every database round trip | plain call | `await` |
| statements (`select`, `insert`, `text`, models) | identical | identical |
| lazy loading of relationships | works (implicit query) | raises; load eagerly or use `run_sync` |
| sync-only libraries (pandas, Alembic, `create_all`) | direct | through `conn.run_sync(...)` |

**Prerequisites**: the pg16 container is up (`docker compose up -d` from the project root) and the
kernel is this project's `.venv`. Jupyter allows top-level `await`, so async cells run as-is.

In [1]:
import asyncio
import os
import pathlib
import sys
import time

from dotenv import load_dotenv

ROOT = next(
    p
    for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
    if (p / "src" / "db.py").exists()
)
sys.path.insert(0, str(ROOT / "src"))
load_dotenv(ROOT / ".env")

from sqlalchemy import ForeignKey, String, create_engine, func, select, text
from sqlalchemy.exc import MissingGreenlet
from sqlalchemy.ext.asyncio import (
    AsyncAttrs,
    async_sessionmaker,
    create_async_engine,
)
from sqlalchemy.orm import (
    DeclarativeBase,
    Mapped,
    mapped_column,
    relationship,
    selectinload,
    sessionmaker,
)

URL = os.environ[
    "DATABASE_URL"
]  # postgresql+psycopg://... ; psycopg 3 serves both APIs

sync_engine = create_engine(URL)
async_engine = create_async_engine(URL)
print(
    "sync :",
    sync_engine.dialect.name,
    sync_engine.dialect.driver,
    "|",
    sync_engine.dialect.is_async,
)
print(
    "async:",
    async_engine.dialect.name,
    async_engine.dialect.driver,
    "|",
    async_engine.dialect.is_async,
)

sync : postgresql psycopg | False
async: postgresql psycopg | True


One URL, two engines. psycopg 3 implements both the blocking and the asyncio DBAPI, which is why the
project defaults to it: the app can be async while Alembic and pandas stay sync, all on one driver.
With `postgresql+asyncpg://` only the async engine would work; with `postgresql+psycopg2://` only the
sync one.

## 1. Core: connections and transactions

`engine.connect()` gives a connection in a transaction that you commit yourself; `engine.begin()`
commits on success and rolls back on an exception. The async versions are the same objects with
`async with` and `await`.

In [3]:
# sync
with sync_engine.connect() as conn:
    print(conn.execute(text("SELECT current_user, version()")).one())
    conn.commit()  # autobegin: a transaction opened at the first execute; nothing to commit here, but explicit is clear

with sync_engine.begin() as conn:  # commits at the end of the block
    conn.execute(text("SELECT 1"))

('app', 'PostgreSQL 16.15 (Debian 16.15-1.pgdg12+2) on aarch64-unknown-linux-gnu, compiled by gcc (Debian 12.2.0-14+deb12u1) 12.2.0, 64-bit')


In [ ]:
# async: same code shape, every round trip awaited
async with async_engine.connect() as conn:
    print((await conn.execute(text("SELECT current_user, version()"))).one())
    await conn.commit()

async with async_engine.begin() as conn:
    await conn.execute(text("SELECT 1"))

Note what is *not* awaited: `.one()`, `.all()`, `.scalars()` on the result. The rows were already
fetched by `await conn.execute(...)`; reading them is in-memory work. (The exception is
`AsyncConnection.stream()`, which yields rows as they arrive and needs `async for`.)

## 2. ORM: models, sessions, and the lazy-loading trap

Models are declared once and used by both sessions. Two small tables, with a relationship, because
relationships are where sync and async behave differently.

In [4]:
class Base(AsyncAttrs, DeclarativeBase):
    # AsyncAttrs adds `obj.awaitable_attrs.<name>` for explicit awaited loads (used in section 2)
    pass


class Author(Base):
    __tablename__ = "sa_authors"
    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str] = mapped_column(String(50))
    notes: Mapped[list["Note"]] = relationship(
        back_populates="author", cascade="all, delete-orphan"
    )


class Note(Base):
    __tablename__ = "sa_notes"
    id: Mapped[int] = mapped_column(primary_key=True)
    author_id: Mapped[int] = mapped_column(ForeignKey("sa_authors.id"))
    body: Mapped[str] = mapped_column(String(200))
    author: Mapped[Author] = relationship(back_populates="notes")


# create_all is sync-only code. The sync engine calls it directly...
Base.metadata.drop_all(sync_engine)
Base.metadata.create_all(sync_engine)

# ...the async engine hands the sync callable a bridged connection via run_sync.
async with async_engine.begin() as conn:
    await conn.run_sync(
        Base.metadata.create_all
    )  # no-op now, the tables exist; shows the pattern
print("tables:", list(Base.metadata.tables))

tables: ['sa_authors', 'sa_notes']


In [5]:
# sync session: write
SyncSession = sessionmaker(sync_engine)

with (
    SyncSession() as session,
    session.begin(),
):  # begin(): commit on exit, rollback on error
    ada = Author(
        name="Ada", notes=[Note(body="analytical engine"), Note(body="first program")]
    )
    grace = Author(name="Grace", notes=[Note(body="compiler")])
    session.add_all([ada, grace])

# sync session: read, and lazy-load the relationship by just touching it
with SyncSession() as session:
    for author in session.scalars(select(Author).order_by(Author.name)):
        print(
            author.name, "->", [n.body for n in author.notes]
        )  # .notes runs a SELECT on access

Ada -> ['analytical engine', 'first program']
Grace -> ['compiler']


In [ ]:
# async session: write. expire_on_commit=False matters (see below).
AsyncSessionLocal = async_sessionmaker(async_engine, expire_on_commit=False)

async with AsyncSessionLocal() as session, session.begin():
    session.add(Author(name="Linus", notes=[Note(body="kernel")]))

# async session: the same lazy access fails
async with AsyncSessionLocal() as session:
    author = (await session.scalars(select(Author).where(Author.name == "Ada"))).one()
    try:
        author.notes  # noqa: B018
    except MissingGreenlet as e:
        print("MissingGreenlet:", str(e).split(";")[0])

MissingGreenlet: greenlet_spawn has not been called


Why: a lazy load is an *implicit* query fired from a plain attribute access, and there is no place
to put an `await` in `author.notes`. The async session refuses instead of blocking the event loop.
The same applies to attributes expired after `commit()`, which is why `expire_on_commit=False` is
the usual setting for `AsyncSession`: otherwise reading `author.name` after a commit would also try
to refresh from the database.

Three ways out, in order of preference:

In [7]:
async with AsyncSessionLocal() as session:
    # 1. eager load in the query (selectinload: one extra SELECT ... WHERE author_id IN (...))
    stmt = select(Author).options(selectinload(Author.notes)).order_by(Author.name)
    for author in await session.scalars(stmt):
        print(author.name, "->", [n.body for n in author.notes])

    # 2. explicit await on one relationship, via the AsyncAttrs mixin on Base
    author = (await session.scalars(select(Author).where(Author.name == "Grace"))).one()
    notes = await author.awaitable_attrs.notes
    print("awaitable_attrs:", [n.body for n in notes])

    # 3. run a sync function inside the session's greenlet; lazy loads work there
    linus = (await session.scalars(select(Author).where(Author.name == "Linus"))).one()
    bodies = await session.run_sync(lambda sync_session: [n.body for n in linus.notes])
    print("run_sync:", bodies)

Ada -> ['analytical engine', 'first program']
Grace -> ['compiler']
Linus -> ['kernel']
awaitable_attrs: ['compiler']
run_sync: ['kernel']


## 3. What async actually buys: overlapping waits

A single query is not faster async. What changes is what the process does *while* the database is
busy. Ten queries that each take 200 ms on the server:

- sync, one after another: about 2 s;
- async with `asyncio.gather`, one connection each from the pool: about 200 ms;
- sync in a thread pool (`asyncio.to_thread`) gets the same overlap, at the price of a thread per query.

In [8]:
SLEEP = text("SELECT pg_sleep(0.2)")
N = 10


def sync_query() -> None:
    with sync_engine.connect() as conn:
        conn.execute(SLEEP)


async def async_query() -> None:
    async with async_engine.connect() as conn:
        await conn.execute(SLEEP)


t0 = time.perf_counter()
for _ in range(N):
    sync_query()
print(f"sync, sequential      : {time.perf_counter() - t0:.2f}s")

t0 = time.perf_counter()
await asyncio.gather(*(async_query() for _ in range(N)))
print(f"async, gather         : {time.perf_counter() - t0:.2f}s")

t0 = time.perf_counter()
await asyncio.gather(*(asyncio.to_thread(sync_query) for _ in range(N)))
print(f"sync in threads       : {time.perf_counter() - t0:.2f}s")

sync, sequential      : 2.08s
async, gather         : 0.25s
sync in threads       : 0.24s


The pool sets the ceiling: `create_async_engine` defaults to `pool_size=5, max_overflow=10`, so up to
15 queries overlap and the 16th waits. The database is doing the same total work in all three cases;
only the client stopped idling. If your bottleneck is the database's CPU, async changes nothing.

## 4. Sync-only libraries from async code

pandas, Alembic, and a lot of older code take a sync connection. From async code, `run_sync`
hands them one that is bridged onto the async connection; you do not need a second engine.

In [9]:
import pandas as pd

stmt = (
    select(Author.name, func.count(Note.id).label("notes"))
    .join(Note)
    .group_by(Author.name)
    .order_by(Author.name)
)

# sync: pandas takes the engine or a connection directly
print(pd.read_sql(stmt, sync_engine))

# async: pandas gets a sync-looking connection through run_sync
async with async_engine.connect() as conn:
    df = await conn.run_sync(lambda sync_conn: pd.read_sql(stmt, sync_conn))
print(df)

    name  notes
0    Ada      2
1  Grace      1
2  Linus      1
    name  notes
0    Ada      2
1  Grace      1
2  Linus      1


## 5. Which one to use

- **Scripts, notebooks, batch jobs, Alembic migrations**: sync. Simpler stack traces, every library
  works, nothing to bridge.
- **Web services and anything that waits on many databases or APIs at once**: async, with the
  framework's event loop (FastAPI, aiohttp, Lambda handlers that fan out). Eager-load relationships,
  keep `expire_on_commit=False`, and give the pool enough size for your concurrency.
- **Mixed code base**: keep one driver (psycopg 3) and one `DATABASE_URL`, create both engines from it,
  and use `run_sync` at the boundary rather than duplicating logic.

`src/db.py` in this project is the async side of that setup; the sync engine in this notebook is the
other half.

In [10]:
# Cleanup: drop the demo tables and release both pools
Base.metadata.drop_all(sync_engine)
sync_engine.dispose()
await async_engine.dispose()
print("done")

done
